<a href="https://colab.research.google.com/github/BallFord/Iqbal-Miftahul-Fikri_2411531004_ML2526/blob/main/Praktikum%205/TugasCrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [2]:
sub_url = 'https://raw.githubusercontent.com/BallFord/Iqbal-Miftahul-Fikri_2411531004_ML2526/refs/heads/main/Praktikum%205/titanic/gender_submission.csv'
train_url = 'https://raw.githubusercontent.com/BallFord/Iqbal-Miftahul-Fikri_2411531004_ML2526/refs/heads/main/Praktikum%205/titanic/train.csv'
test_url = 'https://raw.githubusercontent.com/BallFord/Iqbal-Miftahul-Fikri_2411531004_ML2526/refs/heads/main/Praktikum%205/titanic/test.csv'

train = pd.read_csv(train_url)
test  = pd.read_csv(test_url)
sub   = pd.read_csv(sub_url)

In [3]:
FEATURES = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

def preprocess(df):
    df = df.copy()
    df['Sex']      = df['Sex'].map({'male': 1, 'female': 0})
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    X = df[FEATURES]
    imputer = SimpleImputer(strategy='median')
    X = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)
    return X


X_train_all = preprocess(train)
y_train_all  = train['Survived'].values

X_test_all = preprocess(test)
y_test_all  = sub['Survived'].values


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_all)
X_test_scaled  = scaler.transform(X_test_all)

In [4]:
model = LogisticRegression(max_iter=1000, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

metrics = {
    'Accuracy' : 'accuracy',
    'Precision': 'precision',
    'Recall'   : 'recall',
    'F1 Score' : 'f1',
    'ROC-AUC'  : 'roc_auc'
}

print('=== K-Fold Cross Validation (5-Fold) pada train.csv ===')
for name, scoring in metrics.items():
    scores = cross_val_score(
        model, X_train_scaled, y_train_all, cv=kf, scoring=scoring
    )
    print(f'{name:10s}: {scores.round(4)} | Mean={scores.mean():.4f} | Std={scores.std():.4f}')

=== K-Fold Cross Validation (5-Fold) pada train.csv ===
Accuracy  : [0.7989 0.7978 0.8427 0.7753 0.7809] | Mean=0.7991 | Std=0.0237
Precision : [0.7714 0.7925 0.8088 0.7097 0.6912] | Mean=0.7547 | Std=0.0463
Recall    : [0.7297 0.6269 0.7857 0.6667 0.7231] | Mean=0.7064 | Std=0.0548
F1 Score  : [0.75   0.7    0.7971 0.6875 0.7068] | Mean=0.7283 | Std=0.0403
ROC-AUC   : [0.8803 0.8191 0.8804 0.803  0.8711] | Mean=0.8508 | Std=0.0330


In [5]:
model.fit(X_train_scaled, y_train_all)


y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]


print('=== Final Holdout Evaluation (test.csv vs gender_submission.csv) ===')
print(f'Accuracy : {accuracy_score(y_test_all,  y_pred):.4f}')
print(f'Precision: {precision_score(y_test_all, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test_all,    y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test_all,        y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test_all,   y_prob):.4f}')

=== Final Holdout Evaluation (test.csv vs gender_submission.csv) ===
Accuracy : 0.9378
Precision: 0.8938
Recall   : 0.9408
F1 Score : 0.9167
ROC-AUC  : 0.9822
